In [1]:
import pandas as pd
import json
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
def load_and_extract_features(filepath, is_train=True):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            session = json.loads(line)
            messages = session['messages']

            texts = [m['text'] for m in messages]
            time_diffs = [m['time_diff'] for m in messages]
            
            row = {
                'session_id': session['session_id'],
                'messages_count': len(messages),
                'total_time': sum(time_diffs),
                'last_time_diff': time_diffs[-1],
                'last_is_reply': messages[-1]['is_reply'],
                'full_text': " ".join(texts),
                'last_text': texts[-1],
                'ends_with_punct': int(texts[-1].strip()[-1] in ['.', '!', '?']) if texts[-1].strip() else 0
            }
            
            if is_train:
                row['target'] = session['target']
                
            data.append(row)
            
    return pd.DataFrame(data)

In [3]:
train_df = load_and_extract_features('/kaggle/input/competitions/cu-intro2ai-nlp-26/train.jsonl', is_train=True)
test_df = load_and_extract_features('/kaggle/input/competitions/cu-intro2ai-nlp-26/test.jsonl', is_train=False)

In [4]:
X = train_df.drop(columns=['session_id', 'target'])
y = train_df['target']

text_features = ['full_text', 'last_text']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [5]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    eval_metric='Logloss',
    text_features=text_features,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=50,
    verbose=100
)

model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val)
)

0:	learn: 0.6499627	test: 0.6498567	best: 0.6498567 (0)	total: 315ms	remaining: 5m 15s
100:	learn: 0.5076122	test: 0.5061859	best: 0.5061859 (100)	total: 23.9s	remaining: 3m 32s
200:	learn: 0.5028181	test: 0.5042407	best: 0.5042407 (200)	total: 46.9s	remaining: 3m 6s
300:	learn: 0.4947697	test: 0.5024830	best: 0.5024830 (300)	total: 1m 10s	remaining: 2m 44s
400:	learn: 0.4873481	test: 0.5022225	best: 0.5021095 (390)	total: 1m 34s	remaining: 2m 20s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5021095138
bestIteration = 390

Shrink model to first 391 iterations.


CatBoostClassifier(early_stopping_rounds=50, eval_metric='Logloss', iterations=1000, learning_rate=0.05, random_seed=42, text_features=['full_text', 'last_text'], verbose=100)

In [6]:
val_probs = model.predict_proba(X_val)[:, 1]

best_threshold = 0.5
best_score = 0

for threshold in np.arange(0.3, 0.9, 0.01):
    val_preds = (val_probs >= threshold).astype(int)
    score = fbeta_score(y_val, val_preds, beta=0.5) 
    if score > best_score:
        best_score = score
        best_threshold = threshold
print(f"Оптимальный порог: {best_threshold:.2f}, F0.5 Score на валидации: {best_score:.4f}")

Оптимальный порог: 0.57, F0.5 Score на валидации: 0.7072


In [7]:
test_X = test_df.drop(columns=['session_id'])
test_probs = model.predict_proba(test_X)[:, 1]

test_preds = (test_probs >= best_threshold).astype(int)

submission = pd.DataFrame({
    'session_id': test_df['session_id'],
    'target': test_preds
})
submission.to_csv('submission.csv', index=False)